## 08 — Send traffic for TrustyAI capture

TrustyAI only records **inference** (`POST …/infer`), not metadata `GET`.

**Critical:** call the predictor on **`:9081`** (KServe **agent** / logger).
The Service is **headless** — `:80` → Connection refused; `:8888` talks to OVMS and **skips** TrustyAI.

```text
http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:9081
```


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import requests
from PIL import Image

MODEL = "muffin-chihuahua"
# Headless Service: use agent :9081 (logger). Do NOT use :80 (refused) or :8888 (bypasses TrustyAI).
EP = "http://muffin-chihuahua-predictor.chihuahua-vs-muffin-jan.svc.cluster.local:9081"
SESSION = requests.Session()


def preprocess(path: Path) -> np.ndarray:
    img = Image.open(path).convert("RGB").resize((224, 224))
    arr = np.asarray(img).astype("float32") / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype="float32")
    std = np.array([0.229, 0.224, 0.225], dtype="float32")
    arr = (arr - mean) / std
    return np.transpose(arr, (2, 0, 1))[None, ...].astype(np.float32)


def infer(path: Path):
    chw = preprocess(path)
    raw = chw.tobytes(order="C")
    header = {
        "inputs": [
            {
                "name": "input",
                "shape": list(chw.shape),
                "datatype": "FP32",
                "parameters": {"binary_data_size": len(raw)},
            }
        ]
    }
    hb = json.dumps(header).encode()
    r = SESSION.post(
        f"{EP}/v2/models/{MODEL}/infer",
        data=hb + raw,
        headers={
            "Content-Type": "application/octet-stream",
            "Inference-Header-Content-Length": str(len(hb)),
        },
        timeout=60,
    )
    # JSON fallback if binary not accepted by logger path
    if r.status_code >= 400:
        payload = {
            "inputs": [
                {
                    "name": "input",
                    "shape": list(chw.shape),
                    "datatype": "FP32",
                    "data": chw.ravel().tolist(),
                }
            ]
        }
        r = SESSION.post(
            f"{EP}/v2/models/{MODEL}/infer",
            headers={"Content-Type": "application/json"},
            data=json.dumps(payload),
            timeout=60,
        )
    print(path.name, r.status_code, f"{r.elapsed.total_seconds():.2f}s")
    r.raise_for_status()
    return r


paths = []
root = Path("/opt/app-root/src/data/muffin-chihuahua/test")
paths += list((root / "muffin").glob("*.jpg"))[:3]
paths += list((root / "chihuahua").glob("*.jpg"))[:3]
ood = Path("/opt/app-root/src/ood")
if ood.exists():
    paths += [p for p in ood.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

print("Sending", len(paths), "inferences to", EP)
t0 = time.perf_counter()
for p in paths:
    infer(p)
print(f"Done in {time.perf_counter()-t0:.1f}s — now check TrustyAI /info on the laptop")